# DQ-08 · reviews.csv
Checks: null rate, duplicates, FK, domain values, temporal (review_date ≥ order_date), customer × product consistency.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
rev    = pd.read_csv('reviews.csv', parse_dates=['review_date'])
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
items  = pd.read_csv('order_items.csv', low_memory=False)
cust   = pd.read_csv('customers.csv')
prods  = pd.read_csv('products.csv')
print(f'Shape: {rev.shape}')
rev.head(3)

## 1. Null rate

In [ ]:
null_counts = rev.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(rev)*100).round(2)}))

## 2. Duplicate review_id

In [ ]:
flag('Duplicate review_id', rev.duplicated(subset='review_id'), rev)

## 3. FK checks

In [ ]:
valid_orders = set(orders['order_id'])
valid_prods  = set(prods['product_id'])
valid_cust   = set(cust['customer_id'])

flag('order_id not in orders',     ~rev['order_id'].isin(valid_orders),  rev)
flag('product_id not in products', ~rev['product_id'].isin(valid_prods), rev)
flag('customer_id not in customers', ~rev['customer_id'].isin(valid_cust), rev)

## 4. Domain values: rating

In [ ]:
flag('Invalid rating (not in 1-5)', ~rev['rating'].isin([1,2,3,4,5]), rev)

## 5. Consistency: customer_id matches the order

In [ ]:
order_cust = orders[['order_id','customer_id']].rename(columns={'customer_id':'expected_cust'})
df = rev.merge(order_cust, on='order_id', how='left')
flag('customer_id ≠ order.customer_id', df['customer_id'] != df['expected_cust'], df)

## 6. Consistency: product_id was actually in that order

In [ ]:
ordered_pairs = set(zip(items['order_id'], items['product_id']))
rev_pairs     = set(zip(rev['order_id'],   rev['product_id']))
bad_pairs     = rev_pairs - ordered_pairs
flag('(order_id, product_id) not in order_items', len(bad_pairs), show_sample=False)
print(f'  Bad pairs (sample): {list(bad_pairs)[:5]}')

## 7. Temporal: review_date ≥ order_date

In [ ]:
df2 = rev.merge(orders[['order_id','order_date']], on='order_id', how='left')
flag('review_date < order_date', df2['review_date'] < df2['order_date'], df2)

gap = (df2['review_date'] - df2['order_date']).dt.days
print(f'\nReview gap (days) — order → review:')
print(gap.describe().round(1))
flag('Review gap > 365 days', gap > 365, df2)
flag('Review gap < 0 days',   gap < 0,   df2)

## 8. Only delivered orders should be reviewed

In [ ]:
delivered_orders = set(orders[orders['order_status']=='delivered']['order_id'])
flag('Review for non-delivered order', ~rev['order_id'].isin(delivered_orders), rev)

## Summary

In [ ]:
summary()